# CE-VAE + TUDA Output-Level Adversarial Adaptation (v2)

Fine-tunes CE-VAE with **TUDA's output-level adversarial adaptation** using conservative hyperparameters.

**What changed from v1 (which catastrophically failed):**
- `disc_start: 1500` (was 0 — random discriminator immediately corrupted the generator)
- `disc_weight: 0.3` (was 0.5 — too aggressive)
- `gdl_loss_weight: 0.0` (was 0.5 — destabilized loss landscape)
- `color_loss_weight: 0.0` (was 0.5 — same issue)
- Loss weights now match the **proven** CE-VAE GAN config exactly

**Baseline scores (CE-VAE epoch 119, no GAN):**
- PSNR: 28.9457 | SSIM: 0.8975 | UIQM: 3.0247 | UCIQE: 0.5640

**Expected:** CE-VAE paper Fig. 5 shows GAN loss adds ~2.5 dB PSNR.

**Requirements:** Kaggle T4/P100 GPU, ~2-3 hours, LSUI dataset

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone the repository and cd into it
!rm -rf ce-vae-tuda-underwater-enhancement 2>/dev/null
!git clone https://github.com/priyanshuharshbodhi1/ce-vae-tuda-underwater-enhancement.git
%cd ce-vae-tuda-underwater-enhancement

# Verify we're in the right directory
import os
assert os.path.exists('main.py'), "ERROR: main.py not found! %cd into the repo failed."
assert os.path.exists('src/models/cevae.py'), "ERROR: src/ not found!"
print(f"\nWorking directory: {os.getcwd()}")
print("Repository structure OK")

In [ ]:
# Install dependencies (wandb NOT installed — CSVLogger used instead to avoid hanging)
!pip install -q albumentations lightning omegaconf opencv-python-headless \
    'jsonargparse[signatures]>=4.27.7' Pillow pytorch-msssim torchmetrics scikit-image termcolor lpips matplotlib

# Completely disable wandb — forces CSVLogger fallback in main.py
import os
os.environ['WANDB_MODE'] = 'disabled'

# Set matplotlib to non-interactive backend (needed for Save & Run / headless mode)
import matplotlib
matplotlib.use('Agg')

print("wandb DISABLED (CSVLogger will be used)")
print("matplotlib backend:", matplotlib.get_backend())

## 2. Dataset Setup

Needs: LSUI dataset (paired) + pre-trained CE-VAE checkpoint (epoch 119).
No unpaired real images needed (unlike feature-level TUDA).

In [ ]:
import os
import glob
from pathlib import Path

# Safety: ensure we're in the repo directory
if not os.path.exists('main.py'):
    for d in ['ce-vae-tuda-underwater-enhancement', '/kaggle/working/ce-vae-tuda-underwater-enhancement']:
        if os.path.exists(os.path.join(d, 'main.py')):
            os.chdir(d)
            print(f"Changed to: {os.getcwd()}")
            break

os.makedirs('data', exist_ok=True)

def link_path(src, dst):
    """Create symlink, skip if dst is a real dir/file."""
    if os.path.exists(dst):
        if os.path.islink(dst): os.unlink(dst)
        else: return
    os.symlink(src, dst)
    print(f'  Linked: {src} -> {dst}')

# ---- Detect environment ----
ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = os.path.exists('/content')
search_root = '/kaggle/input' if ON_KAGGLE else ('/content' if ON_COLAB else '.')
print(f'Environment: {"Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "Local"}')
print(f'Searching for datasets in {search_root}...\n')

# ---- 1. Search for LSUI paired dataset ----
print('[1/3] Looking for LSUI dataset (input + GT)...')
found_lsui = False

lsui_dirs = glob.glob(f'{search_root}/**/LSUI', recursive=True)
if lsui_dirs:
    lsui_root = lsui_dirs[0]
    inp = os.path.join(lsui_root, 'input')
    gt = os.path.join(lsui_root, 'GT')
    if os.path.isdir(inp) and os.path.isdir(gt):
        link_path(inp, 'data/input')
        link_path(gt, 'data/GT')
        found_lsui = True

if not found_lsui:
    input_dirs = glob.glob(f'{search_root}/**/input', recursive=True)
    gt_dirs = glob.glob(f'{search_root}/**/GT', recursive=True)
    if input_dirs and gt_dirs:
        link_path(input_dirs[0], 'data/input')
        link_path(gt_dirs[0], 'data/GT')
        found_lsui = True

if found_lsui:
    n_inp = len(glob.glob('data/input/*.*'))
    n_gt = len(glob.glob('data/GT/*.*'))
    print(f'  Found LSUI: {n_inp} input, {n_gt} GT images')
else:
    print('  WARNING: LSUI dataset not found!')

# ---- 2. Search for checkpoint ----
print('\n[2/3] Looking for CE-VAE checkpoint (.ckpt)...')
ckpt_search = glob.glob(f'{search_root}/**/*.ckpt', recursive=True)
if ckpt_search:
    best_ckpt = ckpt_search[0]
    for c in ckpt_search:
        if 'epoch119' in c or 'cevae' in c.lower():
            best_ckpt = c
            break
    link_path(best_ckpt, 'data/lsui-cevae-epoch119.ckpt')
    print(f'  Found checkpoint: {best_ckpt}')
else:
    print('  WARNING: No .ckpt file found!')

# ---- 3. Generate train/val split file lists ----
print('\n[3/3] Generating train/val splits...')
if os.path.exists('data/input') and os.path.exists('data/GT'):
    inputs = sorted(glob.glob('data/input/**/*.*', recursive=True))
    targets = sorted(glob.glob('data/GT/**/*.*', recursive=True))
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
    inputs = [p for p in inputs if Path(p).suffix.lower() in img_exts]
    targets = [p for p in targets if Path(p).suffix.lower() in img_exts]
    
    if len(inputs) == len(targets) and len(inputs) > 0:
        split_idx = int(0.9 * len(inputs))
        with open('data/LSUI_train_input.txt', 'w') as f: f.write('\n'.join(inputs[:split_idx]))
        with open('data/LSUI_train_target.txt', 'w') as f: f.write('\n'.join(targets[:split_idx]))
        with open('data/LSUI_val_input.txt', 'w') as f: f.write('\n'.join(inputs[split_idx:]))
        with open('data/LSUI_val_target.txt', 'w') as f: f.write('\n'.join(targets[split_idx:]))
        print(f'  Train: {split_idx} pairs, Val: {len(inputs)-split_idx} pairs')
    elif len(inputs) != len(targets):
        print(f'  ERROR: input count ({len(inputs)}) != GT count ({len(targets)})')
    else:
        print(f'  ERROR: No images found in data/input/ or data/GT/')
else:
    print('  ERROR: data/input or data/GT not found')

print(f'\nCWD: {os.getcwd()}')
print('--- Data directory contents ---')
!ls -la data/

In [ ]:
# Verify everything is in place
import os
required_files = [
    'data/lsui-cevae-epoch119.ckpt',
    'data/LSUI_train_input.txt',
    'data/LSUI_train_target.txt',
    'data/LSUI_val_input.txt',
    'data/LSUI_val_target.txt',
]

all_good = True
for f in required_files:
    exists = os.path.exists(f)
    status = 'OK' if exists else 'MISSING'
    print(f"  [{status}] {f}")
    if not exists:
        all_good = False

if all_good:
    print("\nAll files ready! Proceed to training.")
else:
    print("\nSome files are missing. Please upload them before training.")

## 3. Training

Fine-tune CE-VAE with TUDA output-level adversarial adaptation (WGAN-GP).

**v2 conservative hyperparameters (vs v1 which failed):**

| Parameter | v1 (FAILED) | v2 (conservative) | Paper GAN config |
|-----------|-------------|-------------------|------------------|
| disc_start | 0 | 1500 | 1000 |
| disc_weight | 0.5 | 0.3 | 0.5 |
| disc_loss | wgan_gp | wgan_gp | hinge |
| gdl_loss_weight | 0.5 | 0.0 | 0.0 |
| color_loss_weight | 0.5 | 0.0 | 0.0 |

Expected time: **~2-3 hours on T4** (20 epochs)

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'  # No wandb — use CSVLogger

# Safety: make sure we're in the repo directory
if not os.path.exists('main.py'):
    for d in ['ce-vae-tuda-underwater-enhancement', '/kaggle/working/ce-vae-tuda-underwater-enhancement']:
        if os.path.exists(os.path.join(d, 'main.py')):
            os.chdir(d)
            print(f"Changed to: {os.getcwd()}")
            break
    else:
        raise FileNotFoundError("Cannot find main.py! Re-run the 'Clone repository' cell above.")

print(f"CWD: {os.getcwd()}")
print(f"Config exists: {os.path.exists('configs/cevae_output_adapt_v2_lsui.yaml')}")
print(f"Checkpoint exists: {os.path.exists('data/lsui-cevae-epoch119.ckpt')}")
print("Starting training (20 epochs, disc_start=1500, WGAN-GP)...\n")

# Train with TUDA output-level adversarial adaptation v2
# disc_start=1500 means first ~2.3 epochs are reconstruction-only warmup
# Then WGAN-GP discriminator gradually improves output quality
!WANDB_MODE=disabled python main.py -cfg configs/cevae_output_adapt_v2_lsui.yaml --trainer.max_epochs 20

print("\nTraining complete!")

## 4. Evaluation

Compare baseline CE-VAE (epoch 119) vs output-adapted CE-VAE on LSUI val set.

In [ ]:
# Find the best checkpoint from training
import os, glob

# Safety: ensure we're in the repo directory
if not os.path.exists('src/models/cevae.py'):
    for d in ['ce-vae-tuda-underwater-enhancement', '/kaggle/working/ce-vae-tuda-underwater-enhancement']:
        if os.path.exists(os.path.join(d, 'src/models/cevae.py')):
            os.chdir(d)
            break

ckpt_files = sorted(glob.glob('training_logs/*/checkpoints/*.ckpt'))
print("Available checkpoints:")
if not ckpt_files:
    print("  No checkpoints found! Training may not have completed.")
    BEST_CKPT = None
else:
    for c in ckpt_files:
        size_mb = os.path.getsize(c) / (1024**2)
        print(f"  {c} ({size_mb:.1f} MB)")
    BEST_CKPT = [c for c in ckpt_files if 'last' in c][-1] if any('last' in c for c in ckpt_files) else ckpt_files[-1]
    print(f"\nUsing checkpoint: {BEST_CKPT}")

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import torch
import numpy as np
from omegaconf import OmegaConf
from src.build.from_config import instantiate_from_config
from src.metrics import compute as compute_metrics
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_model(config_path, ckpt_path):
    """Load a CE-VAE model from config and checkpoint."""
    config = OmegaConf.load(config_path)
    config.model.params.ckpt_path = None
    model = instantiate_from_config(config.model)
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)['state_dict']
    model_keys = set(model.state_dict().keys())
    sd_filtered = {k: v for k, v in sd.items() if k in model_keys}
    model.load_state_dict(sd_filtered, strict=False)
    return model.eval().to(device)

if BEST_CKPT is None:
    print('Skipping model loading - no checkpoint found.')
    model_baseline = None
    model_adapted = None
else:
    print("Loading baseline CE-VAE (epoch 119, no adversarial loss)...")
    model_baseline = load_model('configs/cevae_E2E_lsui.yaml', 'data/lsui-cevae-epoch119.ckpt')
    print("Loading output-adapted CE-VAE (WGAN-GP adversarial v2)...")
    model_adapted = load_model('configs/cevae_output_adapt_v2_lsui.yaml', BEST_CKPT)
    print("Both models loaded!")

In [ ]:
if model_baseline is None or model_adapted is None:
    print('Skipping evaluation - models were not loaded.')
else:
    from torch.utils.data import DataLoader
    from src.data.image_enhancement import DatasetTestFromImageFileList

    test_dataset = DatasetTestFromImageFileList(
        size=256,
        test_images_list_file='data/LSUI_val_input.txt',
        test_target_images_list_file='data/LSUI_val_target.txt'
    )
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

    def evaluate_model(model, loader, name):
        metrics_sum = {'psnr': 0, 'ssim': 0, 'uiqm': 0, 'uciqe': 0}
        count = 0
        with torch.no_grad():
            for batch in tqdm(loader, desc=f'Evaluating {name}'):
                x = batch['image'].permute(0, 3, 1, 2).float().to(device)
                y = batch['target'].permute(0, 3, 1, 2).float().to(device)
                xrec = model(x)
                y_np = torch.clamp(y, -1, 1).detach().cpu()
                y_np = ((y_np + 1) / 2 * 255).permute(0, 2, 3, 1).numpy().astype(np.uint8)
                xrec_np = torch.clamp(xrec, -1, 1).detach().cpu()
                xrec_np = ((xrec_np + 1) / 2 * 255).permute(0, 2, 3, 1).numpy().astype(np.uint8)
                for rec, gt in zip(xrec_np, y_np):
                    res = compute_metrics(rec, gt)
                    for k in metrics_sum:
                        metrics_sum[k] += res[k]
                    count += 1
        return {k: v / count for k, v in metrics_sum.items()}

    print("Evaluating baseline CE-VAE (no adversarial loss)...")
    baseline_metrics = evaluate_model(model_baseline, test_loader, 'Baseline')

    print("\nEvaluating output-adapted CE-VAE (WGAN-GP v2)...")
    adapted_metrics = evaluate_model(model_adapted, test_loader, 'Output-Adapted v2')

    # Print comparison
    print("\n" + "="*65)
    print("RESULTS: Baseline vs Output-Adapted v2 (TUDA WGAN-GP)")
    print("="*65)
    header = f"{'Metric':<10} {'Baseline':>18} {'+ Output Adapt v2':>18} {'Delta':>10}"
    print(header)
    print("-"*58)

    for k in ['psnr', 'ssim', 'uiqm', 'uciqe']:
        delta = adapted_metrics[k] - baseline_metrics[k]
        arrow = '+' if delta > 0 else ''
        line = f"{k.upper():<10} {baseline_metrics[k]:>18.4f} {adapted_metrics[k]:>18.4f} {arrow}{delta:>9.4f}"
        print(line)

    print("="*65)

In [ ]:
if model_baseline is None or model_adapted is None:
    print('Skipping visual comparison.')
else:
    import matplotlib
    matplotlib.use('Agg')  # Ensure non-interactive backend for headless/Save&Run mode
    import matplotlib.pyplot as plt
    from torch.utils.data import DataLoader

    sample_batch = next(iter(DataLoader(test_dataset, batch_size=4, shuffle=True)))
    x = sample_batch['image'].permute(0, 3, 1, 2).float().to(device)
    y = sample_batch['target'].permute(0, 3, 1, 2).float().to(device)

    with torch.no_grad():
        rec_baseline = model_baseline(x)
        rec_adapted = model_adapted(x)

    def tensor_to_img(t):
        t = torch.clamp(t, -1, 1)
        return ((t + 1) / 2).cpu().permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    titles = ['Input (Degraded)', 'Baseline CE-VAE', 'CE-VAE + TUDA v2', 'Ground Truth']
    for i in range(4):
        imgs = [x[i], rec_baseline[i], rec_adapted[i], y[i]]
        for j, (img, title) in enumerate(zip(imgs, titles)):
            axes[i, j].imshow(tensor_to_img(img))
            if i == 0:
                axes[i, j].set_title(title, fontsize=14, fontweight='bold')
            axes[i, j].axis('off')

    plt.suptitle('CE-VAE Baseline vs. Output-Adapted v2 (TUDA WGAN-GP)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('comparison_results_v2.png', dpi=150, bbox_inches='tight')
    plt.close(fig)  # Close figure to free memory (no display in headless mode)
    print("Saved to comparison_results_v2.png")

In [ ]:
if BEST_CKPT is not None:
    import shutil
    output_ckpt = 'cevae_output_adapted_v2.ckpt'
    shutil.copy(BEST_CKPT, output_ckpt)
    print(f"Final checkpoint saved as: {output_ckpt}")
    print(f"Size: {os.path.getsize(output_ckpt) / (1024**2):.1f} MB")
else:
    print('No checkpoint to save.')